# RAG workshop — День 2: асистент з тренувань (training assistant)

Форк ноутбука «асистент ріелтора», перенацілений на **власний корпус тренувань** у `data/`:
таблиці програми 8.0 (CSV), лог силових тестів (CSV), програми 3.0 і 4.0 (PDF),
протокол антикрихкості (Markdown).

Той самий ланцюг: **скан `data/` → chunking → embeddings → ChromaDB → retrieval → промпт → відповідь із джерелами**.

**Що всередині:**
- **Індексація**: обхід `data/`, chunking, embeddings, `upsert` у Chroma зі **стабільними id** (повторний запуск не дублює);
- **Метадані**: `source` (шлях до файлу) для цитування і фільтра `where`, плюс `kind`, `day`, `week`, `date`, `page`;
- **Retrieval**: `collection.query(query_embeddings=..., n_results=k)`;
- **Generation**: промпт «відповідай лише з контексту» + блок **«Джерела:»** у форматі `[source=…]`;
- **Роутер** `where`, **історія діалогу**, **evaluation** на золотому наборі;
- **Telegram-бот** (опційно) — інтерфейс до асистента з історією по `chat_id`.

> **Головна відмінність від ноутбука ріелтора — обробка CSV.** Таблиці програми 8.0 — це
> експорт з таблиці, де назва вправи займає **дві колонки** (номер підходу + результат),
> а тиждень і дата стоять у колонці `Week` у **різних рядках** одного блоку.
> Правило «один рядок = один документ» тут дає сміття, тому нижче — окремий парсер:
> **один чанк = один тиждень**.

### Примітка щодо запуску

**Локально:**
- Потрібні пакети: `openai`, `chromadb`, `python-dotenv`, `pandas`, `pymupdf`
  (+ `python-telegram-bot`, якщо запускати бота)
- Додайте `OPENAI_API_KEY` у файл `.env` у корені репозиторію (див. `.env.example`)
- `TELEGRAM_BOT_TOKEN` — опційно, лише для секції 05

**У Google Colab:**
- Комірка нижче клонує репозиторій у `/content`
- Додайте `OPENAI_API_KEY` через Secrets (🔑 у лівій панелі)
- `TELEGRAM_BOT_TOKEN` — опційно, туди ж

## 00 — Setup
Встановлюємо бібліотеки, читаємо доступи, задаємо шляхи.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Colab detected. Installing packages...")
    !pip install openai chromadb python-dotenv pandas pymupdf python-telegram-bot -q
else:
    print("💻 Local environment detected.")
    # Локально зазвичай усе вже встановлено; розкоментуйте за потреби:
    # !pip install openai chromadb python-dotenv pandas pymupdf -q

In [ ]:
# === Завантаження файлів з GitHub для Colab ===
from pathlib import Path
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import subprocess
    project_root = Path('/content/CrashCourse_Building_RAG_system')
    print("📥 Downloading project files from GitHub...")
    subprocess.run(["rm", "-rf", str(project_root)], capture_output=True)
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/ingvar-goryainov/CrashCourse_Building_RAG_system.git",
         str(project_root)],
        check=True, capture_output=True,
    )
    print("✓ Done:", project_root)

In [ ]:
import csv
import os
import re
import sys
from pathlib import Path

import chromadb
import fitz  # PyMuPDF — читання PDF
from dotenv import load_dotenv
from openai import OpenAI

IN_COLAB = 'google.colab' in sys.modules

# --- Корінь проєкту: шукаємо теку, всередині якої є Day_2/data ---
if IN_COLAB:
    from google.colab import userdata
    PROJECT_ROOT = Path('/content/CrashCourse_Building_RAG_system')
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
        print("✓ OPENAI_API_KEY loaded from Colab Secrets")
    except Exception as e:
        print(f"⚠️  Could not load OPENAI_API_KEY from Secrets: {e}")
else:
    PROJECT_ROOT = Path.cwd()
    for ancestor in [Path.cwd(), *Path.cwd().parents]:
        if (ancestor / 'Day_2' / 'data').is_dir():
            PROJECT_ROOT = ancestor
            break
    load_dotenv(PROJECT_ROOT / '.env')

DATA_DIR = PROJECT_ROOT / 'Day_2' / 'data'
CHROMA_DIR = PROJECT_ROOT / 'Day_2' / 'chroma_data' / 'training_assistant'
COLLECTION_NAME = 'training_corpus_v1'

EMBED_MODEL = 'text-embedding-3-small'
CHAT_MODEL = 'gpt-4o-mini'

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Не знайдено теку з даними: {DATA_DIR}")

# Клієнт створюємо лише за наявності ключа — індексацію/парсинг можна дивитись і без нього.
_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=_api_key) if _api_key else None

print(f"📂 PROJECT_ROOT : {PROJECT_ROOT}")
print(f"📂 DATA_DIR     : {DATA_DIR}")
print(f"💾 CHROMA_DIR   : {CHROMA_DIR}")
print(f"🔑 OPENAI_API_KEY: {'✓ знайдено' if _api_key else '✗ ВІДСУТНІЙ'}")
if not _api_key:
    print("   Локально: додайте OPENAI_API_KEY у .env в корені репозиторію (шаблон — .env.example)")
    print("   Colab   : додайте OPENAI_API_KEY у Secrets (🔑)")
    print("   Без ключа працює лише секція «Dry run» (парсинг без embeddings).")

### Допоміжні функції: chunking і embeddings

Нарізка з **overlap**, щоб не «різати» важливі межі. Для індексації — **batch**-ембеддинги
(менше HTTP-запитів, ніж по одному чанку).

In [ ]:
# Розбиває довгий текст на шматки однакової довжини з перекриттям (overlap).
def chunker(text: str, chunk_size: int = 700, overlap: int = 100) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        piece = text[start : start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        start += chunk_size - overlap
    return chunks


def _require_client() -> OpenAI:
    if client is None:
        raise RuntimeError(
            "OPENAI_API_KEY відсутній. Додайте ключ у .env (локально) або Secrets (Colab) "
            "і перезапустіть комірку Setup."
        )
    return client


# Повертає векторні представлення для кількох текстів одним запитом до API ембеддингів.
def embed_many(texts: list[str]) -> list[list[float]]:
    if not texts:
        return []
    resp = _require_client().embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]

---
## 01 — Збір корпусу документів з `data/`

1. **Сканування:** рекурсивний обхід `data/`, тип файлу — за розширенням (`.md`, `.csv`, `.pdf`).
2. **Обробка за типом:** текст → чанки → метадані → стабільні id.
3. **Колекція** в ChromaDB.
4. **Цикл** по файлах з `upsert` батчами.

### Крок 1: Сканування — що є в каталозі даних

In [ ]:
SUPPORTED_SUFFIXES = {".md", ".csv", ".pdf"}


# Знаходить усі підтримувані файли в каталозі (рекурсивно, включно з підпапками).
def iter_corpus_files(root: Path) -> list[Path]:
    if not root.is_dir():
        return []
    return [p for p in sorted(root.rglob("*"))
            if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES]


DATA_FILES = iter_corpus_files(DATA_DIR)
print(f"Знайдено файлів: {len(DATA_FILES)}")
for p in DATA_FILES:
    print(f"  {p.relative_to(PROJECT_ROOT)}  →  {p.suffix.lower()}")

### Крок 2: Пофайлова обробка

- **`process_md`** / **`process_pdf`** — файл → текст → `chunker` → id / тексти / метадані.
- **`process_csv`** — диспетчер за **формою** файлу (див. нижче).
- **`upsert_batches`** — ембеддинги батчами і **`upsert`** (ідемпотентно за стабільними id).

#### Чому CSV тут окремий випадок

Таблиці `Training Program-8.0-Day N.csv` — це експорт із Google Sheets:

```
Day 1,,,,,,,,,,,
,,,,,,,,,,,
Week,Push press,,Pull-ups,,Lateral squat,,Zercher deadlift,,Cardio,,Notes
1,1,30kg x 6,1,10kg x 3,1,10,1,30kg x 10,,,
,2,35kg x 6,2,10kg x 3,2,10,2,30kg x 10,,,
10 Nov,3,40kg x 6,3,10kg x 3,3,10,3,35kg x 10,,,
```

Проблеми:
- назва вправи в заголовку займає **дві колонки**: номер підходу + результат;
- **тиждень** стоїть у колонці `Week` лише в першому рядку блоку, **дата** — у третьому;
- окремий рядок (`,2,35kg x 6,...`) **не має ні тижня, ні дати, ні назв вправ**.

Тому: **один чанк = один тижневий блок**, зібраний у читабельний текст. Для лога силових
тестів (`Strength tests …csv`) форма інша — там працює звичне «один рядок = один документ».

#### Розширені метадані (поза `source`)

Мінімум для завдання — `source`. Але для цього корпусу дешево додати ще й
`kind` (`program_week` / `record` / `doc`), `day`, `week`, `date`, `page`:
за ними зручно і фільтрувати (`where={"week": "3"}`), і цитувати.

**Принцип** той самий: у `meta` кладемо лише те, по чому реально потрібен фільтр або
цитування; решта деталей лишається в тексті чанка.

In [ ]:
# Перетворює відносний шлях файлу на безпечний префікс для стабільних id у Chroma.
def _safe_id_base(rel_path: str) -> str:
    return rel_path.replace("/", "__").replace("\\", "__")


# Chroma приймає лише скалярні значення — зводимо все до рядків.
def _str_meta(meta: dict) -> dict:
    return {k: ("" if v is None else str(v)) for k, v in meta.items()}


# Відносний шлях файлу — саме він іде в metadata["source"] і в цитати.
def rel_source(path: Path) -> str:
    return str(path.relative_to(PROJECT_ROOT))

In [ ]:
# ---------- Спільні дрібниці для CSV ----------
DAY_TITLE_RE = re.compile(r"^Day\s*\d+", re.IGNORECASE)


def _clean(cell: str) -> str:
    return (cell or "").replace("\n", " / ").strip()


def _read_rows(path: Path) -> list[list[str]]:
    with path.open(newline="", encoding="utf-8-sig") as fh:
        return list(csv.reader(fh))


def _row_is_empty(row: list[str]) -> bool:
    return all(not (c or "").strip() for c in row)


def _header_spans(header: list[str]) -> list[tuple[str, int, int]]:
    """Непорожня клітинка заголовка в колонці i «володіє» колонками i..(наступна непорожня - 1).

    Одне це правило покриває обидва наявні макети: пари «номер підходу + результат»
    (Day 1) і пари «значення + порожньо» (Day 2).
    """
    cols = [i for i, c in enumerate(header) if (c or "").strip()]
    spans = []
    for n, i in enumerate(cols):
        end = cols[n + 1] - 1 if n + 1 < len(cols) else len(header) - 1
        spans.append((header[i].strip(), i, end))
    return spans


def _span_value(row: list[str], start: int, end: int) -> str:
    parts = [_clean(row[i]) for i in range(start, min(end, len(row) - 1) + 1)]
    parts = [p for p in parts if p]
    if not parts:
        return ""
    if len(parts) == 1 and parts[0].isdigit():
        return ""  # номер підходу, результат якого не записали
    if len(parts) == 2 and parts[0].isdigit():
        return f"{parts[0]}) {parts[1]}"
    return " ".join(parts)

In [ ]:
# ---------- CSV-парсер №1: таблиця програми (один чанк = один тиждень) ----------
def parse_program_csv(path: Path) -> list[dict]:
    rows = _read_rows(path)
    day_title = _clean(rows[0][0]) if rows and rows[0] else ""

    hdr_idx = next(
        (i for i, r in enumerate(rows) if r and _clean(r[0]).lower() == "week"), None
    )
    if hdr_idx is None:
        return []

    field_spans = [s for s in _header_spans(rows[hdr_idx]) if s[1] != 0]  # без колонки Week
    m = re.search(r"(\d+\.\d+)", path.name)
    program = m.group(1) if m else ""

    # Групуємо рядки у тижневі блоки: новий блок — коли в колонці Week ціле число;
    # непорожнє нечислове значення там же — це дата блоку.
    blocks, cur = [], None
    for row in rows[hdr_idx + 1:]:
        if _row_is_empty(row):
            continue
        first = _clean(row[0]) if row else ""
        if first.isdigit():
            cur = {"week": first, "date": "", "rows": []}
            blocks.append(cur)
        elif first and cur is not None:
            cur["date"] = first
        if cur is None:
            continue
        cur["rows"].append(row)

    out = []
    for b in blocks:
        title = f"Training Program {program} — {day_title} — Week {b['week']}"
        if b["date"]:
            title += f" ({b['date']})"
        lines = [title]
        for name, start, end in field_spans:
            vals = [v for v in (_span_value(r, start, end) for r in b["rows"]) if v]
            lines.append(f"{name}: {'; '.join(vals) if vals else '—'}")
        out.append({
            "text": "\n".join(lines),
            "meta": {"kind": "program_week", "program": program,
                     "day": day_title, "week": b["week"], "date": b["date"]},
            "suffix": f"w{b['week']}",
        })
    return out


# ---------- CSV-парсер №2: лог записів (один рядок = один документ) ----------
def parse_records_csv(path: Path) -> list[dict]:
    rows = _read_rows(path)
    # Пропускаємо преамбулу: заголовок — перший рядок із >= 3 непорожніми клітинками.
    hdr_idx = next(
        (i for i, r in enumerate(rows) if sum(1 for c in r if (c or "").strip()) >= 3), None
    )
    if hdr_idx is None:
        return []
    header = [_clean(c) for c in rows[hdr_idx]]

    out = []
    for n, row in enumerate(rows[hdr_idx + 1:], start=1):
        if _row_is_empty(row):
            continue
        pairs = [f"{header[i]}: {_clean(row[i])}"
                 for i in range(min(len(header), len(row)))
                 if header[i] and _clean(row[i])]
        if not pairs:
            continue
        out.append({
            "text": " | ".join(pairs),
            "meta": {"kind": "record", "row": str(n)},
            "suffix": f"r{n}",
        })
    return out

In [ ]:
# ---------- Обробники за типом файлу: повертають (ids, docs, metas) ----------
def process_csv(path: Path) -> tuple[list[str], list[str], list[dict]]:
    src = rel_source(path)
    base = _safe_id_base(src)
    rows = _read_rows(path)
    is_program = bool(rows and rows[0] and DAY_TITLE_RE.match(_clean(rows[0][0])))

    recs = parse_program_csv(path) if is_program else parse_records_csv(path)

    if is_program and not recs:
        # Не впізнали форму — не падаємо, а відкочуємось на нарізку всього файлу.
        print(f"  ⚠️  {path.name}: схоже на таблицю програми, але розібрано 0 тижнів — fallback на chunker")
        text = path.read_text(encoding="utf-8-sig")
        parts = chunker(text)
        return (
            [f"{base}::c{i}" for i in range(len(parts))],
            parts,
            [_str_meta({"source": src, "kind": "doc", "chunk": i}) for i in range(len(parts))],
        )

    ids = [f"{base}::{r['suffix']}" for r in recs]
    docs = [r["text"] for r in recs]
    metas = [_str_meta({"source": src, **r["meta"]}) for r in recs]
    return ids, docs, metas


def process_md(path: Path) -> tuple[list[str], list[str], list[dict]]:
    src = rel_source(path)
    base = _safe_id_base(src)
    parts = chunker(path.read_text(encoding="utf-8"))
    return (
        [f"{base}::c{i}" for i in range(len(parts))],
        parts,
        [_str_meta({"source": src, "kind": "doc", "chunk": i}) for i in range(len(parts))],
    )


def process_pdf(path: Path) -> tuple[list[str], list[str], list[dict]]:
    src = rel_source(path)
    base = _safe_id_base(src)
    ids, docs, metas = [], [], []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            for i, part in enumerate(chunker(page.get_text())):
                ids.append(f"{base}::p{page_no}::c{i}")
                docs.append(part)
                metas.append(_str_meta({"source": src, "kind": "doc",
                                        "page": page_no, "chunk": i}))
    return ids, docs, metas


PROCESSORS = {".csv": process_csv, ".md": process_md, ".pdf": process_pdf}

In [ ]:
# Ембеддинги батчами + upsert у Chroma. Стабільні id ⇒ повторний запуск оновлює, а не дублює.
def upsert_batches(collection, ids: list[str], docs: list[str],
                   metas: list[dict], batch_size: int = 64) -> None:
    for start in range(0, len(docs), batch_size):
        b_ids = ids[start : start + batch_size]
        b_docs = docs[start : start + batch_size]
        b_metas = metas[start : start + batch_size]
        collection.upsert(
            ids=b_ids,
            documents=b_docs,
            embeddings=embed_many(b_docs),
            metadatas=b_metas,
        )

### Dry run: перевіряємо парсинг **без** звернень до API

Ця комірка не викликає OpenAI — вона лише проганяє обробники й показує, що саме
потрапить в індекс. Зручно, щоб побачити помилку розбору до того, як витрачати токени.

In [ ]:
total = 0
sample_week = None

print("Чанки за файлами:\n")
for path in DATA_FILES:
    proc = PROCESSORS.get(path.suffix.lower())
    if proc is None:
        continue
    ids, docs, metas = proc(path)
    total += len(docs)
    print(f"  {len(docs):3d}  {path.relative_to(PROJECT_ROOT)}")
    if sample_week is None:
        for d, m in zip(docs, metas):
            if m.get("kind") == "program_week" and m.get("week") == "3":
                sample_week = (d, m)
                break

print(f"\nВсього чанків: {total}")

if sample_week:
    doc, meta = sample_week
    print("\n" + "=" * 60)
    print("Приклад чанка (тижневий блок):")
    print("=" * 60)
    print(doc)
    print("-" * 60)
    print("meta:", meta)

### Крок 3: ChromaDB — персистентний клієнт

1. **`PersistentClient(path=...)`** — індекс на диску між перезапусками.
2. **`get_or_create_collection(name, metadata={"hnsw:space": "cosine"})`** — метрика для `distances`.
3. **`collection.upsert(...)`** — завантаження/оновлення за стабільними id.
4. **`collection.query(query_embeddings=..., n_results=k, where=...)`** — dense retrieval.
5. **`collection.count()` / `.peek()`** — скільки чанків і що всередині.

Колекція — **окрема** (`training_corpus_v1`) і в **окремій теці**, щоб не змішувати зі старим
індексом ріелтора.

In [ ]:
FORCE_REBUILD = False  # True — повна перебудова колекції

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if FORCE_REBUILD:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print(f"🗑️  Колекцію {COLLECTION_NAME} видалено")
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
print(f"Колекція: {COLLECTION_NAME} | чанків зараз: {collection.count()}")

### Крок 4: цикл індексації

Для кожного файлу — відповідний обробник і одразу `upsert_batches`.
Через стабільні id повторний запуск **не створює дублікатів**.

In [ ]:
if FORCE_REBUILD or collection.count() == 0:
    print("Індексація: обхід файлів у data/ …")
    for path in DATA_FILES:
        proc = PROCESSORS.get(path.suffix.lower())
        if proc is None:
            continue
        ids, docs, metas = proc(path)
        if not docs:
            print("  пропуск (порожній):", path.relative_to(PROJECT_ROOT))
            continue
        upsert_batches(collection, ids, docs, metas)
        print(f"  OK {path.relative_to(PROJECT_ROOT)} → {len(docs)} чанків")
    print("\nГотово.")
else:
    print("Колекція вже заповнена — пропускаємо індексацію (FORCE_REBUILD=True для перебудови).")

print("collection.count() =", collection.count())

In [ ]:
# Що саме лежить в індексі
peek = collection.peek(limit=3)
for doc, meta in zip(peek["documents"], peek["metadatas"]):
    print(meta)
    print(doc[:200].replace("\n", " ") + ("…" if len(doc) > 200 else ""))
    print("---")

---
## 02 — Retrieval

`retrieve` — типовий **dense retrieval**: рахуємо вектор запиту тією ж моделлю `EMBED_MODEL`,
що й при індексації, і викликаємо `collection.query` з **`query_embeddings`**, **`n_results=k`**,
опційним **`where`** і `include=["documents", "distances", "metadatas"]`.

Колекція створена з `hnsw:space="cosine"`, тому в `distances` — косинусна відстань
(менше = ближче).

In [ ]:
# Повертає top-k найближчих чанків за косинусною відстанню (за потреби з фільтром метаданих where).
def retrieve(query: str, k: int = 6, where: dict | None = None):
    q_emb = embed_many([query])[0]
    return collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        where=where,
        include=["documents", "distances", "metadatas"],
    )


def show(res) -> None:
    for dist, doc, meta in zip(res["distances"][0], res["documents"][0], res["metadatas"][0]):
        print(f"[{dist:.3f}] {meta.get('source')}  ({meta.get('kind')})")
        print("  " + doc[:280].replace("\n", " | ") + ("…" if len(doc) > 280 else ""))
        print("---")


show(retrieve("Скільки я робив push press на 10 тижні?", k=5))

In [ ]:
show(retrieve("Який мій результат у присіді на силовому тесті?", k=5))

### Фільтр `where` (за файлом — `source`)

У метаданих кожного чанка є `source` — відносний шлях до файлу. Пошук можна обмежити
одним джерелом (або кількома через `$in`), а також фільтрувати за `kind` / `week` / `day`.

In [ ]:
# Лише чанки з одного файлу
show(retrieve(
    "Що я робив на кардіо?",
    k=5,
    where={"source": "Day_2/data/Training Program-8.0-Day 2.csv"},
))

In [ ]:
# Фільтр не тільки за файлом: лише тижневі блоки 3-го тижня по всіх днях
show(retrieve(
    "Які ваги я брав?",
    k=5,
    where={"$and": [{"kind": "program_week"}, {"week": "3"}]},
))

---
## 03 — Augmentation + Generation

### Промпт-інжиніринг для RAG

Модель **не має доступу** до всієї бази — лише до **відібраних фрагментів**. Промпт має
керувати тим, **як читати контекст**, **коли відмовлятися** і **як цитувати**.

**Кейс 1 (базовий):** роль + межі + заборона вигадувати + обовʼязковий блок **«Джерела:»**
списком рядків **`[source=…]`** (шлях такий самий, як у заголовку фрагмента контексту).
Retrieval — по всьому корпусу, без `where`.

In [ ]:
PROMPT_TEMPLATE = """Ти персональний асистент із силових тренувань. Відповідай українською.

Правила:
- Спирайся **лише** на наведений контекст. Якщо даних недостатньо або їх немає — скажи це прямо
  і не вигадуй.
- **Ніколи** не вигадуй назву вправи, вагу, кількість повторів або дату, якої немає в контексті.
- Числа (ваги, повтори, тижні, дати) переписуй точно так, як вони стоять у контексті.
- **Якщо в контексті немає відповіді** — прямо напиши, що таких даних у матеріалах немає,
  і **не додавай** блок «Джерела:» взагалі: цитувати нема чого. Не підставляй замість
  відповіді схожий, але інший тиждень / вправу / дату.
- У кінці додай блок **«Джерела:»** — маркований список рядків **точно у форматі заголовків
  контексту**: `[source=…]`, де шлях такий самий, як у рядку перед фрагментом
  (наприклад `[source=Day_2/data/Training Program-8.0-Day 1.csv]`).
- Якщо факт узятий із фрагмента CSV, джерелом усе одно вказуй **файл**, а не номер рядка чи тижня.

Контекст:
{context}

Питання: {question}
"""


# Збирає знайдені чанки в текст контексту із заголовками [source=…] перед кожним фрагментом.
def build_context(res) -> str:
    blocks = []
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        blocks.append(f"[source={meta.get('source')}]\n{doc}")
    return "\n\n".join(blocks)


def rag_answer(question: str, k: int = 6, where: dict | None = None) -> str:
    res = retrieve(question, k=k, where=where)
    prompt = PROMPT_TEMPLATE.format(context=build_context(res), question=question)
    resp = _require_client().chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

In [ ]:
print(rag_answer("Скільки я жав у push press на 10 тижні Day 1?"))

In [ ]:
# Перевірка на галюцинацію: у корпусі немає даних про марафон.
print(rag_answer("Який у мене результат на марафоні 2025 року?"))

### Кейс 2: маршрутизація `where` через промпт

Додаємо **крок роутера**: невеликий промпт дає моделі **список дозволених `source`**
(рівно тих, що проіндексовані), і вона повертає один/кілька шляхів або `null`.
Відповідь **парситься і фільтрується по allowlist** — вигаданий шлях не пройде.
Для кількох файлів — `{"source": {"$in": [...]}}`.

In [ ]:
# Ті самі відносні шляхи, що потрапили в індекс (унеможливлює вигадані source)
ALLOWED_SOURCES = sorted({rel_source(p) for p in DATA_FILES})

ROUTER_PROMPT = """Ти маєш визначити, які файли зі списку потрібні для відповіді на питання.
Дозволені шляхи source (обирай лише з цього списку):
{sources_list}

Питання користувача: {question}

Підказки:
- тижні, підходи, ваги в програмі 8.0, конкретний день (Day 1/2/3) → відповідний
  "Training Program-8.0-Day N.csv"
- силові тести, максимуми, 1ПМ, власна вага, дати тестів → "Strength tests Training Program 8.0.csv"
- профілактика травм, розминка, антикрихкість, спина/коліна/плечі → "Antifragility protocol.md"
- склад програми 3.0 або 4.0, перелік вправ по днях → відповідний PDF
- якщо питання загальне або незрозуміло, який файл потрібен → поверни null

Відповідь: лише шляхи (по одному в рядку) або слово null. Без пояснень.
"""


def route_sources(question: str) -> list[str]:
    prompt = ROUTER_PROMPT.format(
        sources_list="\n".join(f"- {s}" for s in ALLOWED_SOURCES),
        question=question,
    )
    resp = _require_client().chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = (resp.choices[0].message.content or "").strip()
    if raw.lower().startswith("null"):
        return []
    picked = []
    for line in re.split(r"[\n,]", raw):
        cand = line.strip().lstrip("-").strip()
        if cand in ALLOWED_SOURCES and cand not in picked:  # allowlist
            picked.append(cand)
    return picked


def where_from_sources(sources: list[str]) -> dict | None:
    if not sources:
        return None
    if len(sources) == 1:
        return {"source": sources[0]}
    return {"source": {"$in": sources}}


def rag_answer_with_router(question: str, k: int = 6) -> str:
    picked = route_sources(question)
    print(f"[router] → {picked or 'null (пошук по всьому корпусу)'}")
    return rag_answer(question, k=k, where=where_from_sources(picked))

In [ ]:
question = "Які вправи входять у протокол антикрихкості?"

print("=== Case 1: без роутера ===")
print(rag_answer(question))
print("\n=== Case 2: з роутером ===")
print(rag_answer_with_router(question))

### Кейс 3: памʼятаємо діалог

**Історія** — ланцюжок останніх реплік у межах сесії. Вона допомагає зрозуміти уточнення
(«а на 5 тижні?»), але **не замінює retrieval**: факти щоразу беруться з Chroma за поточним
питанням, а історія лише додається в промпт.

Зберігаємо в памʼяті процесу (`dict[chat_id]`), лишаємо вікно останніх 4 обмінів,
щоб не роздувати промпт.

In [ ]:
CHAT_HISTORY: dict[int, list[dict]] = {}
MAX_HISTORY_TURNS = 4  # скільки пар user+assistant лишати


def trim_history(messages: list[dict], max_turns: int = MAX_HISTORY_TURNS) -> list[dict]:
    return messages[-(2 * max_turns):]


def get_history(chat_id: int) -> list[dict]:
    return CHAT_HISTORY.get(chat_id, [])


def append_history(chat_id: int, question: str, answer: str) -> None:
    msgs = CHAT_HISTORY.get(chat_id, []) + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    CHAT_HISTORY[chat_id] = trim_history(msgs)


def rag_answer_with_router_history(question: str, k: int = 6, chat_id: int = 0) -> str:
    picked = route_sources(question)
    res = retrieve(question, k=k, where=where_from_sources(picked))
    prompt = PROMPT_TEMPLATE.format(context=build_context(res), question=question)

    messages = get_history(chat_id) + [{"role": "user", "content": prompt}]
    resp = _require_client().chat.completions.create(
        model=CHAT_MODEL, messages=messages, temperature=0,
    )
    answer = resp.choices[0].message.content
    append_history(chat_id, question, answer)
    return answer

In [ ]:
DEMO_CHAT_ID = 9001


def run_dialog(turns: list[str], chat_id: int = DEMO_CHAT_ID) -> None:
    CHAT_HISTORY.pop(chat_id, None)  # чиста сесія перед демо
    for i, q in enumerate(turns, 1):
        print(f"\n{'=' * 60}\nTurn {i}: {q}\n{'=' * 60}")
        print(rag_answer_with_router_history(q, k=5, chat_id=chat_id))


run_dialog([
    "Скільки я робив zercher deadlift на 3 тижні Day 1?",
    "а на 10 тижні?",
])

---
## 04 — Evaluation

**Evaluation** — перевірка, що система працює на **ваших** даних і питаннях. Збираємо
**золотий набір**: питання + які файли мали потрапити в цитати + прийнятна відповідь.

Дві метрики нижче:
- **Source coverage** — чи всі очікувані файли зʼявилися в блоці «Джерела:»
  (`|cited ∩ expected| / |expected|`);
- **LLM-суддя** — чи містить відповідь ті самі ключові факти, що й еталон.

Останній кейс (`E06`) — **проба на галюцинацію**: у корпусі немає таких даних, і правильна
поведінка — відмова, а не вигадана відповідь.

In [ ]:
EVAL_SET = [
    {
        "id": "E01",
        "question": "Скільки я жав у push press на 10 тижні Day 1?",
        "expected_sources": ["Day_2/data/Training Program-8.0-Day 1.csv"],
        "final_answer": "На 10 тижні (12 Jan) push press: 52.5kg x 5, 52.5kg x 5, 55kg x 4, 55kg x 4.",
    },
    {
        "id": "E02",
        "question": "Який мій результат у back squat на силовому тесті?",
        "expected_sources": ["Day_2/data/Strength tests Training Program 8.0.csv"],
        "final_answer": "Back squat — 110 кг на 3 повтори, 26 Jan 2026, за власної ваги 99 кг.",
    },
    {
        "id": "E03",
        "question": "Які вправи входять у протокол антикрихкості і як часто його робити?",
        "expected_sources": ["Day_2/data/Antifragility protocol.md"],
        "final_answer": (
            "Jefferson curl, side bend, sissy squat, shoulder external rotation, "
            "neck extension iso — виконувати перед тренуванням двічі на тиждень."
        ),
    },
    {
        "id": "E04",
        "question": "Які вправи в Day 2 програми 4.0?",
        "expected_sources": ["Day_2/data/Training Program 4.0.pdf"],
        "final_answer": (
            "Block power clean, Romanian Deadlift | Deadlift, Dips | Push-ups, Body saw."
        ),
    },
    {
        "id": "E05",
        "question": "Яке кардіо я робив на 2 тижні Day 2?",
        "expected_sources": ["Day_2/data/Training Program-8.0-Day 2.csv"],
        "final_answer": "23 хвилини бігу із середнім пульсом 146 bpm.",
    },
    {
        "id": "E06",  # проба на галюцинацію — у корпусі цього немає
        "question": "Який у мене результат на марафоні 2025 року?",
        "expected_sources": [],
        "final_answer": "У наданих матеріалах немає даних про марафон — відповісти неможливо.",
    },
]

In [ ]:
JUDGE_PROMPT = """Ти суддя якості RAG. Чи містить відповідь ACTUAL ті самі ключові факти,
що й REFERENCE (допускається інше формулювання)?
Відповідь лише одним словом: YES або NO.

Питання: {question}

REFERENCE:
{reference}

ACTUAL:
{actual}
"""


def extract_sources(answer: str) -> set[str]:
    """Витягує шляхи з блоку «Джерела:» у форматі [source=…]."""
    return set(re.findall(r"\[source=([^\]]+)\]", answer or ""))


def judge_matches_reference(question: str, reference: str, actual: str) -> bool:
    resp = _require_client().chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            question=question, reference=reference, actual=actual)}],
        temperature=0,
    )
    return (resp.choices[0].message.content or "").strip().upper().startswith("YES")


def run_eval(rag_fn=rag_answer, k: int = 6) -> None:
    cov_total, judge_total = 0.0, 0
    for case in EVAL_SET:
        answer = rag_fn(case["question"], k=k)
        cited = extract_sources(answer)
        expected = set(case["expected_sources"])

        # Для проби на галюцинацію «покриття» = чи утримались від зайвих цитат.
        coverage = (len(cited & expected) / len(expected)) if expected else (1.0 if not cited else 0.0)
        ok = judge_matches_reference(case["question"], case["final_answer"], answer)

        cov_total += coverage
        judge_total += int(ok)

        print(f"\n{'=' * 60}\n{case['id']}: {case['question']}")
        print(f"  coverage: {coverage:.2f}  | judge: {'YES' if ok else 'NO'}")
        print(f"  cited   : {sorted(cited) or '—'}")
        print(f"  expected: {sorted(expected) or '— (очікується відмова)'}")
        print(f"  answer  : {answer[:300]}{'…' if len(answer) > 300 else ''}")

    n = len(EVAL_SET)
    print(f"\n{'=' * 60}")
    print(f"Source coverage (avg): {cov_total / n:.2f}")
    print(f"Judge accuracy       : {judge_total}/{n} = {judge_total / n:.2f}")


run_eval(rag_answer)

In [ ]:
# Те саме, але з роутером — порівняйте coverage і точність
run_eval(rag_answer_with_router)

---
## 05 — Telegram-бот (опційно)

Потрібен пакет **`python-telegram-bot`**. Токен: **`TELEGRAM_BOT_TOKEN`** у `.env`
(локально) або в Secrets (Colab).

Перед запуском виконайте комірки **роутера (Кейс 2)** і **історії (Кейс 3)** — бот викликає
**`rag_answer_with_router_history`** з `chat_id` Telegram-чату, тож кожен чат має власну
історію в `CHAT_HISTORY` (у памʼяті процесу).

Команди: **`/reset`** — очистити історію діалогу в цьому чаті.

> Бот **опційний**: якщо токена немає, комірки нижче просто повідомлять про це і нічого
> не запустять — решта ноутбука від цього не залежить.

In [ ]:
# --- Токен бота ---
if IN_COLAB:
    try:
        os.environ["TELEGRAM_BOT_TOKEN"] = userdata.get('TELEGRAM_BOT_TOKEN')
        print("✓ TELEGRAM_BOT_TOKEN loaded from Colab Secrets")
    except Exception as e:
        print(f"⚠️  Could not load TELEGRAM_BOT_TOKEN from Secrets: {e}")
else:
    load_dotenv(PROJECT_ROOT / '.env')

BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")

if BOT_TOKEN:
    print("🔑 TELEGRAM_BOT_TOKEN: ✓ знайдено")
else:
    # Не кидаємо виняток: бот опційний, решта ноутбука працює без нього.
    print("🔑 TELEGRAM_BOT_TOKEN: ✗ ВІДСУТНІЙ — бот не запуститься")
    print("   Локально: додайте TELEGRAM_BOT_TOKEN у .env (шаблон — .env.example)")
    print("   Colab   : додайте TELEGRAM_BOT_TOKEN у Secrets (🔑)")
    print("   Токен видає @BotFather у Telegram")

In [ ]:
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters

app = globals().get("app")  # щоб повторний запуск комірки не лишав два інстанси

if not BOT_TOKEN:
    print("⚠️  TELEGRAM_BOT_TOKEN не встановлений. Бот не буде запущено.")
else:
    # Якщо комірка запускається повторно — коректно зупиняємо попередній інстанс.
    if app is not None:
        for stop_step in ("updater_stop", "stop", "shutdown"):
            try:
                if stop_step == "updater_stop":
                    await app.updater.stop()
                elif stop_step == "stop":
                    await app.stop()
                else:
                    await app.shutdown()
            except Exception:
                pass

    async def cmd_reset(update, context):
        """Очищує історію CHAT_HISTORY для цього чату."""
        CHAT_HISTORY.pop(update.effective_chat.id, None)
        await update.message.reply_text("Історію діалогу очищено. Можете питати знову.")

    async def cmd_start(update, context):
        await update.message.reply_text(
            "Привіт! Я асистент по твоїх тренуваннях.\n\n"
            "Спитай, наприклад:\n"
            "• Скільки я жав у push press на 10 тижні Day 1?\n"
            "• Який мій результат у back squat на силовому тесті?\n"
            "• Які вправи в протоколі антикрихкості?\n\n"
            "/reset — очистити історію діалогу."
        )

    async def handle(update, context):
        if not update.message or not update.message.text:
            return
        chat_id = update.effective_chat.id
        question = update.message.text.strip()
        # Роутер + RAG + історія: get_history → LLM → append_history (Кейс 3)
        try:
            answer = rag_answer_with_router_history(question, k=5, chat_id=chat_id)
            await update.message.reply_text(answer)
        except Exception as e:
            await update.message.reply_text(f"Помилка: {str(e)[:200]}")

    app = ApplicationBuilder().token(BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", cmd_start))
    app.add_handler(CommandHandler("reset", cmd_reset))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle))

    await app.initialize()
    await app.start()
    await app.updater.start_polling()
    print("✅ Telegram-бот запущено (з історією по chat_id). /reset — очистити діалог.")
    print("💡 Зупинка: запустіть комірку нижче")

In [ ]:
# Зупинка бота (запустіть, тільки якщо він працює)
app = globals().get("app")

if app is not None and BOT_TOKEN:
    try:
        await app.updater.stop()
        await app.stop()
        await app.shutdown()
        app = None
        print("✅ Telegram-бот зупинено")
    except Exception as e:
        print(f"⚠️  Помилка при зупинці бота: {e}")
else:
    print("Бот не запущено або токен відсутній")

---
## Підсумок: як вимоги закриті

| # | Вимога | Де в ноутбуці |
|---|---|---|
| 1 | Обхід теки → chunking → embeddings → ChromaDB (`upsert`, стабільні id) | `iter_corpus_files`, `chunker`, `embed_many`, `upsert_batches`, id виду `…::w3` / `…::r4` / `…::p1::c0` |
| 2 | Метадані з `source` для цитування і `where` | `_str_meta({"source": …, "kind": …, "day": …, "week": …})` |
| 3 | `collection.query` з `query_embeddings` і параметром `k` | `retrieve(query, k=6, where=None)` |
| 4 | Промпт «лише з контексту» + блок «Джерела:» у форматі `[source=…]` | `PROMPT_TEMPLATE`, `build_context`, `rag_answer` |
| 5 | Відтворювана логіка | цей `.ipynb`; `FORCE_REBUILD` для чистої перебудови |

**Ідемпотентність:** id детерміновані (шлях + позиція), тож повторний прогін індексації
робить `upsert` поверх тих самих записів — `collection.count()` не росте.